In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/src/enrich

In [0]:
import pytest
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date
from decimal import Decimal

In [0]:
# Test DataFrames

fact_orders = spark.createDataFrame(
    [("C1", "P1", 1, "O1", date(2024,3,15), date(2024,3,20), "Standard", 2, 100.0, 0.1, Decimal("10.00"), 0),
     ("C2", "P2", 2, "O2", date(2024,1,1),  date(2024,1,5),  "Express",  1, 50.0,  0.0, Decimal("5.00"),  1),
     ("C1", "P2", 3, "O3", date(2025,6,10), date(2025,6,15), "Standard", 3, 200.0, 0.2, Decimal("20.00"), 2)],
    ["customer_id", "product_id", "row_id", "order_id", "order_date", "ship_date",
     "ship_mode", "quantity", "price", "discount", "profit", "order_key"]
)

dim_customers = spark.createDataFrame(
    [("C1", "Alice", "US", 0),
     ("C2", "Bob",   "UK", 1)],
    ["customer_id", "customer_name", "country", "customer_key"]
)

dim_products = spark.createDataFrame(
    [("P1", "Chair", "Furniture", "Seats", 0),
     ("P2", "Phone", "Tech",      "Mobile", 1)],
    ["product_id", "product_name", "category", "sub_category", "product_key"]
)

In [0]:
# Test enriched orders join produces correct columns
enriched_orders = build_enriched_orders(fact_orders, dim_customers, dim_products)

def test_enriched_has_expected_columns():
    expected = ["customer_name", "country", "category", "sub_category"]
    for col in expected:
        assert col in enriched_orders.columns, f"Missing column: {col}"

def test_enriched_keeps_order_columns():
    for col in ["order_id", "order_date", "ship_date", "profit", "quantity"]:
        assert col in enriched_orders.columns, f"Missing order column: {col}"

In [0]:
# Test inner join drops unmatched rows

def test_enriched_row_count():
    assert enriched_orders.count() == 3

def test_unmatched_customer_dropped():
    extra_order = spark.createDataFrame(
        [("C99", "P1", 4, "O4", date(2024,1,1), date(2024,1,5), "Std", 1, 10.0, 0.0, Decimal("1.00"), 3)],
        fact_orders.columns
    )
    df = fact_orders.union(extra_order)
    result = build_enriched_orders(df, dim_customers, dim_products)
    assert result.count() == 3

def test_unmatched_product_dropped():
    extra_order = spark.createDataFrame(
        [("C1", "P99", 5, "O5", date(2024,1,1), date(2024,1,5), "Std", 1, 10.0, 0.0, Decimal("1.00"), 4)],
        fact_orders.columns
    )
    df = fact_orders.union(extra_order)
    result = build_enriched_orders(df, dim_customers, dim_products)
    assert result.count() == 3

In [0]:
# Test no nulls in enriched join columns

def test_no_null_customer_name():
    assert enriched_orders.filter(F.col("customer_name").isNull()).count() == 0

def test_no_null_category():
    assert enriched_orders.filter(F.col("category").isNull()).count() == 0

In [0]:
# Test deduplication works

def test_enriched_no_duplicates():
    total = enriched_orders.count()
    distinct = enriched_orders.dropDuplicates().count()
    assert total == distinct

In [0]:
# Test profit aggregation

def test_profit_by_year_columns():
    profit_by_year = build_profit_by_year(enriched_orders)
    expected = ["year", "category", "sub_category", "customer_name", "total_profit"]
    assert profit_by_year.columns == expected

def test_profit_by_year_values():
    profit_by_year = build_profit_by_year(enriched_orders)
    row = profit_by_year.filter(
        (F.col("year") == 2024) & (F.col("customer_name") == "Alice") & (F.col("category") == "Furniture")
    ).collect()[0]
    assert float(row["total_profit"]) == 10.0

def test_profit_by_year_groups_by_year():
    profit_by_year = build_profit_by_year(enriched_orders)
    years = [r["year"] for r in profit_by_year.select("year").distinct().collect()]
    assert 2024 in years
    assert 2025 in years

In [0]:
# Test enriched_orders table vs source tables row count

def test_enriched_table_row_count():
    fact_df = spark.read.table("az_ci_de_dbs.ecom_dev.fact_orders")
    enriched_df = spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
    assert enriched_df.count() <= fact_df.count()
    assert enriched_df.count() > 0

def test_profit_by_year_table_not_empty():
    profit_df = spark.read.table("az_ci_de_dbs.ecom_dev.profit_by_year")
    assert profit_df.count() > 0

In [0]:
# Run all tests

test_functions = [
    test_enriched_has_expected_columns,
    test_enriched_keeps_order_columns,
    test_enriched_row_count,
    test_unmatched_customer_dropped,
    test_unmatched_product_dropped,
    test_no_null_customer_name,
    test_no_null_category,
    test_enriched_no_duplicates,
    test_profit_by_year_columns,
    test_profit_by_year_values,
    test_profit_by_year_groups_by_year,
    test_enriched_table_row_count,
    test_profit_by_year_table_not_empty,
]

passed, failed = 0, 0
for fn in test_functions:
    try:
        fn()
        passed += 1
        print(f"  [PASSED] {fn.__name__}")
    except Exception as e:
        failed += 1
        print(f" [FAILED] {fn.__name__}: {e}")

print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")